In [ ]:
# -*- coding: utf-8 -*-
"""
Modelo corregido de la interacción TNF-α, IL-6, IL-10 en inflamación post-ictus.

Basado en la descripción del artículo de Arishi et al. (2026) PLOS ONE.
Correcciones implementadas:
- Inducción de IL-10 mediante función de Hill con n=2 y retraso temporal (τ=18 h).
- Términos de supresión de TNF-α e IL-6 por IL-10 (lineales, como en el original).
- Parámetros ajustables mediante widgets interactivos.

Autor: Generado por IA para análisis crítico.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# ----------------------------------------------------------------------
# Definición del modelo corregido (Ecuaciones Diferenciales Ordinarias)
# ----------------------------------------------------------------------
def model_corregido(t, y, params, tau=18.0):
    """
    y = [T, I6, I]  # TNF-α, IL-6, IL-10
    params = (gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
              delta_T, delta_6, delta_I, phi_T, phi_6,
              K, n, beta_T, beta_6)
    """
    T, I6, I = y
    (gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
     delta_T, delta_6, delta_I, phi_T, phi_6,
     K, n, beta_T, beta_6) = params

    # -- Ecuación para TNF-α (T) --
    # Producción basal + activación por IL-6 - degradación - supresión por IL-10
    dT_dt = gamma_T + alpha_6 * I6 - delta_T * T - phi_T * I * T

    # -- Ecuación para IL-6 (I6) --
    # Producción basal + activación por TNF-α - degradación - supresión por IL-10
    dI6_dt = gamma_6 + alpha_T * T - delta_6 * I6 - phi_6 * I * I6

    # -- Ecuación para IL-10 (I) con retraso y Hill n=2 --
    # La producción se activa solo si t >= tau, y depende de (T + I6) con cooperatividad.
    if t >= tau:
        S = T + I6
        # Función de Hill con coeficiente n
        produccion_IL10 = gamma_I * (S**n) / (K**n + S**n)
    else:
        produccion_IL10 = 0.0

    # Degradación de IL-10 (primer orden)
    dI_dt = produccion_IL10 - delta_I * I

    return [dT_dt, dI6_dt, dI_dt]

# ----------------------------------------------------------------------
# Parámetros base (tomados de la Tabla 1 del artículo, con ajustes menores)
# ----------------------------------------------------------------------
#   gamma_T : producción basal TNF-α       (nM/h)
#   gamma_6 : producción basal IL-6        (nM/h)
#   gamma_I : producción máxima IL-10      (nM/h)
#   alpha_T : activación de TNF-α por IL-6 (1/h)
#   alpha_6 : activación de IL-6 por TNF-α (1/h)
#   delta_T : degradación TNF-α            (1/h)
#   delta_6 : degradación IL-6             (1/h)
#   delta_I : degradación IL-10            (1/h)
#   phi_T   : supresión de TNF-α por IL-10 (1/(nM·h))
#   phi_6   : supresión de IL-6 por IL-10  (1/(nM·h))
#   K       : constante de activación media para IL-10 (nM)
#   n       : coeficiente de Hill (cooperatividad)
#   beta_T, beta_6 : no se usan en este modelo (los dejamos por compatibilidad)

parametros_base = (0.05,    # gamma_T
                   0.02,    # gamma_6
                   0.8,     # gamma_I   (producción de IL-10)
                   0.15,    # alpha_T   (IL-6 -> TNF-α)
                   0.12,    # alpha_6   (TNF-α -> IL-6)
                   0.25,    # delta_T
                   0.20,    # delta_6
                   0.15,    # delta_I
                   0.08,    # phi_T     (supresión TNF-α)
                   0.05,    # phi_6     (supresión IL-6)
                   0.5,     # K         (cte de Hill)
                   2.0,     # n         (coeficiente de Hill)
                   0.0,     # beta_T    (no usado)
                   0.0)     # beta_6    (no usado)

# ----------------------------------------------------------------------
# Condiciones iniciales (como en el artículo)
# ----------------------------------------------------------------------
T0 = 0.1      # nM
I60 = 0.05    # nM
I0 = 0.01     # nM
y0 = [T0, I60, I0]

# Tiempo de simulación: 72 horas
t_span = (0, 72)
t_eval = np.linspace(0, 72, 500)

# ----------------------------------------------------------------------
# Función para resolver y graficar con parámetros modificables
# ----------------------------------------------------------------------
def simular_y_graficar(gamma_T=0.05, gamma_6=0.02, gamma_I=0.8,
                       alpha_T=0.15, alpha_6=0.12,
                       delta_T=0.25, delta_6=0.20, delta_I=0.15,
                       phi_T=0.08, phi_6=0.05,
                       K=0.5, n=2.0,
                       tau=18.0,
                       T0=0.1, I60=0.05, I0=0.01):

    params = (gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
              delta_T, delta_6, delta_I, phi_T, phi_6,
              K, n, 0.0, 0.0)

    y0_local = [T0, I60, I0]

    # Resolver el sistema
    sol = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                    t_span, y0_local, t_eval=t_eval, method='RK45')

    T_sim, I6_sim, I_sim = sol.y

    # Crear figura
    plt.figure(figsize=(12, 6))
    plt.plot(sol.t, T_sim, 'r-', linewidth=2, label='TNF-α')
    plt.plot(sol.t, I6_sim, 'b-', linewidth=2, label='IL-6')
    plt.plot(sol.t, I_sim, 'g-', linewidth=2, label='IL-10')

    # Sombrear regiones (opcional, para comparar con Fig 4)
    plt.axvspan(6, 24, alpha=0.1, color='gray', label='Ventana proinflamatoria (6-24h)')
    plt.axvspan(36, 60, alpha=0.1, color='lightgreen', label='Ventana de resolución (36-60h)')

    # Línea vertical en el retraso (τ)
    plt.axvline(tau, color='black', linestyle='--', linewidth=1, label=f'Retraso IL-10 (t={tau}h)')

    plt.xlabel('Tiempo (horas)', fontsize=12)
    plt.ylabel('Concentración (nM)', fontsize=12)
    plt.title('Dinámica corregida de citocinas en inflamación post-ictus', fontsize=14)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.ylim(bottom=0)  # Para evitar valores negativos
    plt.tight_layout()
    plt.show()

# ----------------------------------------------------------------------
# Interfaz interactiva con widgets
# ----------------------------------------------------------------------
widgets.interact(simular_y_graficar,
                 gamma_T=widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.005, description='γ_T (prod TNF)'),
                 gamma_6=widgets.FloatSlider(value=0.02, min=0.0, max=0.1, step=0.002, description='γ_6 (prod IL-6)'),
                 gamma_I=widgets.FloatSlider(value=0.8, min=0.0, max=2.0, step=0.05, description='γ_I (prod IL-10)'),
                 alpha_T=widgets.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.01, description='α_T (IL-6→TNF)'),
                 alpha_6=widgets.FloatSlider(value=0.12, min=0.0, max=0.5, step=0.01, description='α_6 (TNF→IL-6)'),
                 delta_T=widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.01, description='δ_T (degr TNF)'),
                 delta_6=widgets.FloatSlider(value=0.20, min=0.05, max=0.8, step=0.01, description='δ_6 (degr IL-6)'),
                 delta_I=widgets.FloatSlider(value=0.15, min=0.05, max=0.5, step=0.01, description='δ_I (degr IL-10)'),
                 phi_T=widgets.FloatSlider(value=0.08, min=0.0, max=0.3, step=0.005, description='φ_T (sup TNF)'),
                 phi_6=widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.005, description='φ_6 (sup IL-6)'),
                 K=widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description='K (Hill)'),
                 n=widgets.FloatSlider(value=2.0, min=1.0, max=4.0, step=0.1, description='n (coef Hill)'),
                 tau=widgets.FloatSlider(value=18.0, min=0.0, max=48.0, step=1.0, description='τ (retraso, h)'),
                 T0=widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.01, description='TNF₀'),
                 I60=widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='IL-6₀'),
                 I0=widgets.FloatSlider(value=0.01, min=0.0, max=0.1, step=0.001, description='IL-10₀'))

interactive(children=(FloatSlider(value=0.05, description='γ_T (prod TNF)', max=0.2, step=0.005), FloatSlider(…

<function __main__.simular_y_graficar(gamma_T=0.05, gamma_6=0.02, gamma_I=0.8, alpha_T=0.15, alpha_6=0.12, delta_T=0.25, delta_6=0.2, delta_I=0.15, phi_T=0.08, phi_6=0.05, K=0.5, n=2.0, tau=18.0, T0=0.1, I60=0.05, I0=0.01)>

In [2]:
# -*- coding: utf-8 -*-
"""
Código completo corregido para replicar el artículo de Arishi et al. (2026).
Incluye todas las simulaciones y análisis con 12 parámetros.
"""

# Instalación de SALib (si no está instalado)
!pip install SALib

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
from SALib.sample import saltelli
from SALib.analyze import sobol
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------
# MODELO CORREGIDO (12 PARÁMETROS, SIN beta_T ni beta_6)
# ----------------------------------------------------------------------
def model_corregido(t, y, params, tau):
    """
    y = [T, I6, I]  # TNF-α, IL-6, IL-10
    params = (gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
              delta_T, delta_6, delta_I, phi_T, phi_6, K, n)
    """
    T, I6, I = y
    (gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
     delta_T, delta_6, delta_I, phi_T, phi_6, K, n) = params

    # TNF-α
    dT_dt = gamma_T + alpha_6 * I6 - delta_T * T - phi_T * I * T

    # IL-6
    dI6_dt = gamma_6 + alpha_T * T - delta_6 * I6 - phi_6 * I * I6

    # IL-10: producción con retraso y Hill
    if t >= tau:
        S = T + I6
        produccion_IL10 = gamma_I * (S**n) / (K**n + S**n)
    else:
        produccion_IL10 = 0.0

    dI_dt = produccion_IL10 - delta_I * I

    return [dT_dt, dI6_dt, dI_dt]

# ----------------------------------------------------------------------
# PARÁMETROS BASE Y CONDICIONES INICIALES
# ----------------------------------------------------------------------
parametros_base = {
    'gamma_T': 0.05,
    'gamma_6': 0.02,
    'gamma_I': 0.8,
    'alpha_T': 0.15,
    'alpha_6': 0.12,
    'delta_T': 0.25,
    'delta_6': 0.20,
    'delta_I': 0.15,
    'phi_T': 0.08,
    'phi_6': 0.05,
    'K': 0.5,
    'n': 2.0
}
tau_base = 18.0
y0_base = [0.1, 0.05, 0.01]   # TNF-α, IL-6, IL-10

# Tiempo de simulación
t_span = (0, 72)
t_eval = np.linspace(0, 72, 500)

# ----------------------------------------------------------------------
# FUNCIONES DE SIMULACIÓN (cada una recibe lista de 12 parámetros)
# ----------------------------------------------------------------------
def simular_basal(params_list, y0, tau, t_span, t_eval, ax=None):
    """Simulación basal."""
    sol = solve_ivp(lambda t, y: model_corregido(t, y, params_list, tau),
                    t_span, y0, t_eval=t_eval, method='RK45')
    T, I6, I = sol.y
    if ax is not None:
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.axvspan(6, 24, alpha=0.1, color='gray', label='Ventana proinflamatoria')
        ax.axvspan(36, 60, alpha=0.1, color='lightgreen', label='Ventana resolución')
        ax.axvline(tau, color='k', ls='--', lw=1, label=f'Retraso τ={tau}h')
        ax.set_xlabel('Tiempo (h)')
        ax.set_ylabel('Concentración (nM)')
        ax.set_title('Simulación basal')
        ax.legend(loc='upper right')
        ax.grid(alpha=0.3)
    return sol.t, T, I6, I

def simular_bolus_IL10(params_list, y0, tau, t_span, t_eval, ax=None,
                       dosis=0.05, tiempo_dosis=24):
    """Simulación con dosis de IL-10."""
    def model_with_bolus(t, y):
        if abs(t - tiempo_dosis) < 1e-2:
            y = list(y)
            y[2] += dosis
        return model_corregido(t, y, params_list, tau)

    sol = solve_ivp(lambda t, y: model_with_bolus(t, y),
                    t_span, y0, t_eval=t_eval, method='RK45', max_step=0.1)
    T, I6, I = sol.y
    if ax is not None:
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.axvline(tiempo_dosis, color='orange', ls=':', lw=2, label=f'Dosis IL-10 (t={tiempo_dosis}h)')
        ax.set_xlabel('Tiempo (h)')
        ax.set_ylabel('Concentración (nM)')
        ax.set_title(f'Dosis de IL-10 ({dosis} nM a las {tiempo_dosis} h)')
        ax.legend(loc='upper right')
        ax.grid(alpha=0.3)
    return sol.t, T, I6, I

def simular_inhibicion_TNF(params_list, y0, tau, t_span, t_eval, ax=None,
                           factor_inhibicion=0.5):
    """Inhibición de TNF-α (reduce gamma_T y alpha_6)."""
    gamma_T, gamma_6, gamma_I, alpha_T, alpha_6, delta_T, delta_6, delta_I, phi_T, phi_6, K, n = params_list
    gamma_T_inh = gamma_T * factor_inhibicion
    alpha_6_inh = alpha_6 * factor_inhibicion
    params_inh = (gamma_T_inh, gamma_6, gamma_I, alpha_T, alpha_6_inh,
                  delta_T, delta_6, delta_I, phi_T, phi_6, K, n)

    sol = solve_ivp(lambda t, y: model_corregido(t, y, params_inh, tau),
                    t_span, y0, t_eval=t_eval, method='RK45')
    T, I6, I = sol.y
    if ax is not None:
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.set_xlabel('Tiempo (h)')
        ax.set_ylabel('Concentración (nM)')
        ax.set_title(f'Inhibición TNF-α ({factor_inhibicion*100:.0f}%)')
        ax.legend(loc='upper right')
        ax.grid(alpha=0.3)
    return sol.t, T, I6, I

# ----------------------------------------------------------------------
# ANÁLISIS DE BIFURCACIÓN (Figura 2)
# ----------------------------------------------------------------------
def bifurcacion_phi_T(params_dict, y0_pro, y0_res, tau, t_final=500, num_pasos=50):
    """Barre phi_T y obtiene estados estacionarios."""
    phi_T_range = np.linspace(0.02, 0.15, num_pasos)
    T_pro, T_res = [], []

    for phi_T in phi_T_range:
        params = (params_dict['gamma_T'], params_dict['gamma_6'], params_dict['gamma_I'],
                  params_dict['alpha_T'], params_dict['alpha_6'],
                  params_dict['delta_T'], params_dict['delta_6'], params_dict['delta_I'],
                  phi_T, params_dict['phi_6'], params_dict['K'], params_dict['n'])

        # Desde proinflamatorio
        sol_pro = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                            (0, t_final), y0_pro, method='RK45', t_eval=[t_final])
        T_pro.append(sol_pro.y[0, -1])

        # Desde resolutivo
        sol_res = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                            (0, t_final), y0_res, method='RK45', t_eval=[t_final])
        T_res.append(sol_res.y[0, -1])

    plt.figure(figsize=(8,5))
    plt.plot(phi_T_range, T_pro, 'b.-', label='Rama proinflamatoria')
    plt.plot(phi_T_range, T_res, 'g.-', label='Rama resolutiva')
    plt.axvline(x=0.08, color='r', linestyle='--', label='Punto de bifurcación (aprox)')
    plt.xlabel('φ_T (supresión TNF-α por IL-10) [1/(nM·h)]')
    plt.ylabel('TNF-α estacionario (nM)')
    plt.title('Diagrama de bifurcación variando φ_T')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# ----------------------------------------------------------------------
# DIAGRAMA DE DISPERSIÓN (Figura 3)
# ----------------------------------------------------------------------
def diagrama_dispersion(params_dict, y0_res, tau, phi_T_range=(0.08, 0.15), num=30):
    """Correlación en la rama resolutiva."""
    phi_vals = np.linspace(phi_T_range[0], phi_T_range[1], num)
    T_vals = []

    for phi_T in phi_vals:
        params = (params_dict['gamma_T'], params_dict['gamma_6'], params_dict['gamma_I'],
                  params_dict['alpha_T'], params_dict['alpha_6'],
                  params_dict['delta_T'], params_dict['delta_6'], params_dict['delta_I'],
                  phi_T, params_dict['phi_6'], params_dict['K'], params_dict['n'])
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                        (0, 500), y0_res, method='RK45', t_eval=[500])
        T_vals.append(sol.y[0, -1])

    # Ajuste lineal
    def lineal(x, a, b):
        return a * x + b
    popt, _ = curve_fit(lineal, phi_vals, T_vals)
    a, b = popt
    T_pred = lineal(phi_vals, a, b)
    ss_res = np.sum((T_vals - T_pred)**2)
    ss_tot = np.sum((T_vals - np.mean(T_vals))**2)
    r2 = 1 - ss_res/ss_tot

    plt.figure(figsize=(6,5))
    plt.scatter(phi_vals, T_vals, color='green', label='Datos')
    plt.plot(phi_vals, T_pred, 'k--', label=f'Ajuste lineal (R² = {r2:.3f})')
    plt.xlabel('φ_T')
    plt.ylabel('TNF-α estacionario (nM)')
    plt.title('Rama resolutiva: correlación φ_T vs TNF-α')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    print(f"Ecuación: TNF-α = {a:.4f} * φ_T + {b:.4f}")
    print(f"R² = {r2:.4f}")

# ----------------------------------------------------------------------
# ANÁLISIS DE SENSIBILIDAD GLOBAL (Sobol, Tabla 2)
# ----------------------------------------------------------------------
def analisis_sensibilidad(params_dict, y0, tau, t_final=72, N=100):
    """Índices de Sobol para 5 parámetros clave."""
    problema = {
        'num_vars': 5,
        'names': ['gamma_I', 'phi_T', 'phi_6', 'alpha_T', 'alpha_6'],
        'bounds': [[0.2, 1.5], [0.02, 0.15], [0.01, 0.1], [0.05, 0.3], [0.05, 0.25]]
    }
    param_values = saltelli.sample(problema, N, calc_second_order=False)
    print(f"Evaluando {len(param_values)} simulaciones...")

    Y = np.zeros(len(param_values))
    for i, vals in enumerate(param_values):
        gamma_I, phi_T, phi_6, alpha_T, alpha_6 = vals
        params = (params_dict['gamma_T'], params_dict['gamma_6'], gamma_I,
                  alpha_T, alpha_6,
                  params_dict['delta_T'], params_dict['delta_6'], params_dict['delta_I'],
                  phi_T, phi_6, params_dict['K'], params_dict['n'])
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                        (0, t_final), y0, method='RK45', t_eval=[t_final])
        Y[i] = sol.y[0, -1]

    Si = sobol.analyze(problema, Y, calc_second_order=False, print_to_console=False)

    df_sens = pd.DataFrame({
        'Parámetro': problema['names'],
        'S1 (efecto principal)': Si['S1'],
        'ST (efecto total)': Si['ST'],
        'ST_conf': Si['ST_conf']
    })
    print("\nÍndices de Sobol (efecto sobre TNF-α final):")
    print(df_sens.to_string(index=False))

    plt.figure(figsize=(8,4))
    x = np.arange(len(problema['names']))
    plt.bar(x - 0.2, Si['S1'], 0.4, label='S1 (principal)', alpha=0.7)
    plt.bar(x + 0.2, Si['ST'], 0.4, label='ST (total)', alpha=0.7)
    plt.xticks(x, problema['names'])
    plt.ylabel('Índice de Sobol')
    plt.title('Análisis de sensibilidad global')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    return Si

# ----------------------------------------------------------------------
# INTERFAZ INTERACTIVA CON PESTAÑAS
# ----------------------------------------------------------------------
# Widgets para modificar parámetros
style = {'description_width': 'initial'}
sliders = {
    'gamma_T': widgets.FloatSlider(value=parametros_base['gamma_T'], min=0, max=0.2, step=0.005, description='γ_T (prod TNF)'),
    'gamma_6': widgets.FloatSlider(value=parametros_base['gamma_6'], min=0, max=0.1, step=0.002, description='γ_6 (prod IL-6)'),
    'gamma_I': widgets.FloatSlider(value=parametros_base['gamma_I'], min=0, max=2.0, step=0.05, description='γ_I (prod IL-10)'),
    'alpha_T': widgets.FloatSlider(value=parametros_base['alpha_T'], min=0, max=0.5, step=0.01, description='α_T (IL-6→TNF)'),
    'alpha_6': widgets.FloatSlider(value=parametros_base['alpha_6'], min=0, max=0.5, step=0.01, description='α_6 (TNF→IL-6)'),
    'delta_T': widgets.FloatSlider(value=parametros_base['delta_T'], min=0.05, max=0.8, step=0.01, description='δ_T (degr TNF)'),
    'delta_6': widgets.FloatSlider(value=parametros_base['delta_6'], min=0.05, max=0.8, step=0.01, description='δ_6 (degr IL-6)'),
    'delta_I': widgets.FloatSlider(value=parametros_base['delta_I'], min=0.05, max=0.5, step=0.01, description='δ_I (degr IL-10)'),
    'phi_T': widgets.FloatSlider(value=parametros_base['phi_T'], min=0.0, max=0.3, step=0.005, description='φ_T (sup TNF)'),
    'phi_6': widgets.FloatSlider(value=parametros_base['phi_6'], min=0.0, max=0.2, step=0.005, description='φ_6 (sup IL-6)'),
    'K': widgets.FloatSlider(value=parametros_base['K'], min=0.1, max=2.0, step=0.1, description='K (Hill)'),
    'n': widgets.FloatSlider(value=parametros_base['n'], min=1.0, max=4.0, step=0.1, description='n (Hill)'),
    'tau': widgets.FloatSlider(value=tau_base, min=0, max=48, step=1, description='τ (retraso, h)'),
    'T0': widgets.FloatSlider(value=y0_base[0], min=0, max=0.5, step=0.01, description='TNF₀'),
    'I60': widgets.FloatSlider(value=y0_base[1], min=0, max=0.5, step=0.01, description='IL-6₀'),
    'I0': widgets.FloatSlider(value=y0_base[2], min=0, max=0.1, step=0.001, description='IL-10₀')
}

def ejecutar_simulacion(tipo, **kwargs):
    """Función unificada que llama al análisis correspondiente."""
    # Recolectar valores actuales de los sliders
    params_dict = {k: sliders[k].value for k in ['gamma_T','gamma_6','gamma_I','alpha_T','alpha_6',
                                                   'delta_T','delta_6','delta_I','phi_T','phi_6','K','n']}
    tau = sliders['tau'].value
    y0 = [sliders['T0'].value, sliders['I60'].value, sliders['I0'].value]

    # Convertir a lista para las funciones que lo requieren
    params_list = [params_dict[k] for k in ['gamma_T','gamma_6','gamma_I','alpha_T','alpha_6',
                                             'delta_T','delta_6','delta_I','phi_T','phi_6','K','n']]

    if tipo == 'basal':
        fig, ax = plt.subplots(figsize=(10,5))
        simular_basal(params_list, y0, tau, t_span, t_eval, ax)
        plt.show()
    elif tipo == 'bolus':
        dosis = kwargs.get('dosis', 0.05)
        t_dosis = kwargs.get('t_dosis', 24)
        fig, ax = plt.subplots(figsize=(10,5))
        simular_bolus_IL10(params_list, y0, tau, t_span, t_eval, ax, dosis, t_dosis)
        plt.show()
    elif tipo == 'inhibicion':
        factor = kwargs.get('factor', 0.5)
        fig, ax = plt.subplots(figsize=(10,5))
        simular_inhibicion_TNF(params_list, y0, tau, t_span, t_eval, ax, factor)
        plt.show()
    elif tipo == 'bifurcacion':
        y0_pro = [0.5, 0.3, 0.01]   # TNF alto, IL-10 bajo
        y0_res = [0.05, 0.03, 0.5]  # TNF bajo, IL-10 alto
        bifurcacion_phi_T(params_dict, y0_pro, y0_res, tau)
    elif tipo == 'dispersion':
        diagrama_dispersion(params_dict, [0.05, 0.03, 0.5], tau)
    elif tipo == 'sensibilidad':
        analisis_sensibilidad(params_dict, y0, tau, t_final=72, N=50)

# Crear pestañas
tab = widgets.Tab()
children = []

# Pestaña 0: Basal
out_basal = widgets.interactive_output(ejecutar_simulacion, {'tipo': widgets.fixed('basal')})
children.append(out_basal)

# Pestaña 1: IL-10 bolus
widgets_bolus = {'tipo': widgets.fixed('bolus'),
                 'dosis': widgets.FloatSlider(value=0.05, min=0, max=0.2, step=0.01, description='Dosis (nM)'),
                 't_dosis': widgets.FloatSlider(value=24, min=0, max=72, step=1, description='Tiempo dosis (h)')}
out_bolus = widgets.interactive_output(ejecutar_simulacion, widgets_bolus)
children.append(out_bolus)

# Pestaña 2: Inhibición TNF
widgets_inh = {'tipo': widgets.fixed('inhibicion'),
               'factor': widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Factor inhibición')}
out_inh = widgets.interactive_output(ejecutar_simulacion, widgets_inh)
children.append(out_inh)

# Pestaña 3: Bifurcación
out_bif = widgets.interactive_output(ejecutar_simulacion, {'tipo': widgets.fixed('bifurcacion')})
children.append(out_bif)

# Pestaña 4: Dispersión
out_disp = widgets.interactive_output(ejecutar_simulacion, {'tipo': widgets.fixed('dispersion')})
children.append(out_disp)

# Pestaña 5: Sensibilidad
out_sens = widgets.interactive_output(ejecutar_simulacion, {'tipo': widgets.fixed('sensibilidad')})
children.append(out_sens)

tab.children = children
tab.set_title(0, 'Basal')
tab.set_title(1, 'IL-10 bolus')
tab.set_title(2, 'Inhibición TNF')
tab.set_title(3, 'Bifurcación')
tab.set_title(4, 'Dispersión')
tab.set_title(5, 'Sensibilidad')

# Mostrar todo
display(widgets.VBox([widgets.HBox(list(sliders.values())), tab]))

In [3]:
# -*- coding: utf-8 -*-
"""
Código completo con parámetros interactivos actualizables.
Todos los sliders están conectados a la función de simulación.
"""

# Instalación de SALib (si no está instalado)
#!pip install SALib

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
from SALib.sample import saltelli
from SALib.analyze import sobol
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------
# MODELO CORREGIDO (12 PARÁMETROS)
# ----------------------------------------------------------------------
def model_corregido(t, y, params, tau):
    T, I6, I = y
    (gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
     delta_T, delta_6, delta_I, phi_T, phi_6, K, n) = params

    dT_dt = gamma_T + alpha_6 * I6 - delta_T * T - phi_T * I * T
    dI6_dt = gamma_6 + alpha_T * T - delta_6 * I6 - phi_6 * I * I6

    if t >= tau:
        S = T + I6
        produccion_IL10 = gamma_I * (S**n) / (K**n + S**n)
    else:
        produccion_IL10 = 0.0

    dI_dt = produccion_IL10 - delta_I * I
    return [dT_dt, dI6_dt, dI_dt]

# ----------------------------------------------------------------------
# FUNCIÓN PRINCIPAL QUE RECIBE TODOS LOS PARÁMETROS COMO ARGUMENTOS
# ----------------------------------------------------------------------
def ejecutar_simulacion(
    # Parámetros del modelo
    gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
    delta_T, delta_6, delta_I, phi_T, phi_6, K, n,
    tau, T0, I60, I0,
    # Tipo de simulación y argumentos específicos
    tipo='basal',
    dosis=0.05, tiempo_dosis=24, factor_inhibicion=0.5
):
    # Construir lista de parámetros
    params_list = [gamma_T, gamma_6, gamma_I, alpha_T, alpha_6,
                   delta_T, delta_6, delta_I, phi_T, phi_6, K, n]
    y0 = [T0, I60, I0]
    t_span = (0, 72)
    t_eval = np.linspace(0, 72, 500)

    # Figura
    fig, ax = plt.subplots(figsize=(10, 5))

    if tipo == 'basal':
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params_list, tau),
                        t_span, y0, t_eval=t_eval, method='RK45')
        T, I6, I = sol.y
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.axvspan(6, 24, alpha=0.1, color='gray', label='Ventana proinflamatoria')
        ax.axvspan(36, 60, alpha=0.1, color='lightgreen', label='Ventana resolución')
        ax.axvline(tau, color='k', ls='--', lw=1, label=f'Retraso τ={tau}h')
        ax.set_title('Simulación basal')

    elif tipo == 'bolus':
        def model_with_bolus(t, y):
            if abs(t - tiempo_dosis) < 1e-2:
                y = list(y)
                y[2] += dosis
            return model_corregido(t, y, params_list, tau)
        sol = solve_ivp(lambda t, y: model_with_bolus(t, y),
                        t_span, y0, t_eval=t_eval, method='RK45', max_step=0.1)
        T, I6, I = sol.y
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.axvline(tiempo_dosis, color='orange', ls=':', lw=2, label=f'Dosis IL-10 (t={tiempo_dosis}h)')
        ax.set_title(f'Dosis de IL-10 ({dosis} nM a las {tiempo_dosis} h)')

    elif tipo == 'inhibicion':
        gamma_T_inh = gamma_T * factor_inhibicion
        alpha_6_inh = alpha_6 * factor_inhibicion
        params_inh = [gamma_T_inh, gamma_6, gamma_I, alpha_T, alpha_6_inh,
                      delta_T, delta_6, delta_I, phi_T, phi_6, K, n]
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params_inh, tau),
                        t_span, y0, t_eval=t_eval, method='RK45')
        T, I6, I = sol.y
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.set_title(f'Inhibición TNF-α ({factor_inhibicion*100:.0f}%)')

    ax.set_xlabel('Tiempo (h)')
    ax.set_ylabel('Concentración (nM)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
    plt.show()

# ----------------------------------------------------------------------
# CREACIÓN DE LOS WIDGETS (SLIDERS) PARA CADA PARÁMETRO
# ----------------------------------------------------------------------
sliders = {
    'gamma_T':   widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.005, description='γ_T (prod TNF)'),
    'gamma_6':   widgets.FloatSlider(value=0.02, min=0.0, max=0.1, step=0.002, description='γ_6 (prod IL-6)'),
    'gamma_I':   widgets.FloatSlider(value=0.8,  min=0.0, max=2.0, step=0.05, description='γ_I (prod IL-10)'),
    'alpha_T':   widgets.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.01, description='α_T (IL-6→TNF)'),
    'alpha_6':   widgets.FloatSlider(value=0.12, min=0.0, max=0.5, step=0.01, description='α_6 (TNF→IL-6)'),
    'delta_T':   widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.01, description='δ_T (degr TNF)'),
    'delta_6':   widgets.FloatSlider(value=0.20, min=0.05, max=0.8, step=0.01, description='δ_6 (degr IL-6)'),
    'delta_I':   widgets.FloatSlider(value=0.15, min=0.05, max=0.5, step=0.01, description='δ_I (degr IL-10)'),
    'phi_T':     widgets.FloatSlider(value=0.08, min=0.0, max=0.3, step=0.005, description='φ_T (sup TNF)'),
    'phi_6':     widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.005, description='φ_6 (sup IL-6)'),
    'K':         widgets.FloatSlider(value=0.5,  min=0.1, max=2.0, step=0.1, description='K (Hill)'),
    'n':         widgets.FloatSlider(value=2.0,  min=1.0, max=4.0, step=0.1, description='n (Hill)'),
    'tau':       widgets.FloatSlider(value=18.0, min=0.0, max=48.0, step=1.0, description='τ (retraso, h)'),
    'T0':        widgets.FloatSlider(value=0.1,  min=0.0, max=0.5, step=0.01, description='TNF₀'),
    'I60':       widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='IL-6₀'),
    'I0':        widgets.FloatSlider(value=0.01, min=0.0, max=0.1, step=0.001, description='IL-10₀')
}

# Organizar los sliders en un layout de varias filas (5 columnas)
slider_items = list(sliders.items())
# Crear filas de 5 sliders cada una
slider_rows = []
for i in range(0, len(slider_items), 5):
    row_widgets = [slider_items[j][1] for j in range(i, min(i+5, len(slider_items)))]
    slider_rows.append(widgets.HBox(row_widgets))
slider_panel = widgets.VBox(slider_rows)

# ----------------------------------------------------------------------
# PESTAÑAS CON DIFERENTES TIPOS DE SIMULACIÓN
# ----------------------------------------------------------------------
# Definir los argumentos comunes que siempre se pasan (todos los sliders)
args_comunes = {name: slider for name, slider in sliders.items()}

# Pestaña basal: solo tipo='basal'
out_basal = widgets.interactive_output(
    ejecutar_simulacion,
    {**args_comunes, 'tipo': widgets.fixed('basal')}
)

# Pestaña bolus: añadir widgets para dosis y tiempo
dosis_widget = widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.01, description='Dosis (nM)')
tiempo_dosis_widget = widgets.FloatSlider(value=24, min=0, max=72, step=1, description='Tiempo dosis (h)')
out_bolus = widgets.interactive_output(
    ejecutar_simulacion,
    {**args_comunes,
     'tipo': widgets.fixed('bolus'),
     'dosis': dosis_widget,
     'tiempo_dosis': tiempo_dosis_widget}
)

# Pestaña inhibición: añadir factor
factor_widget = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Factor inhibición')
out_inhibicion = widgets.interactive_output(
    ejecutar_simulacion,
    {**args_comunes,
     'tipo': widgets.fixed('inhibicion'),
     'factor_inhibicion': factor_widget}
)

# Pestañas para bifurcación, dispersión y sensibilidad (requieren funciones especiales)
# Para estas, no usamos la función general, sino las específicas (no interactivas con sliders)
# Las dejamos como botones que ejecutan el análisis con los valores actuales de los sliders.

def crear_pestanya_no_interactiva(titulo, funcion):
    """Crea una pestaña con un botón que ejecuta la función con los valores actuales de los sliders."""
    button = widgets.Button(description=f"Ejecutar {titulo}")
    output = widgets.Output()

    def on_button_clicked(b):
        with output:
            clear_output(wait=True)
            # Recolectar valores actuales
            params_dict = {k: sliders[k].value for k in ['gamma_T','gamma_6','gamma_I','alpha_T','alpha_6',
                                                          'delta_T','delta_6','delta_I','phi_T','phi_6','K','n']}
            tau = sliders['tau'].value
            y0 = [sliders['T0'].value, sliders['I60'].value, sliders['I0'].value]
            # Llamar a la función correspondiente
            if titulo == 'Bifurcación':
                y0_pro = [0.5, 0.3, 0.01]
                y0_res = [0.05, 0.03, 0.5]
                funcion(params_dict, y0_pro, y0_res, tau)
            elif titulo == 'Dispersión':
                funcion(params_dict, [0.05, 0.03, 0.5], tau)
            elif titulo == 'Sensibilidad':
                funcion(params_dict, y0, tau, t_final=72, N=50)

    button.on_click(on_button_clicked)
    return widgets.VBox([button, output])

# Definir las funciones de análisis (copiadas de la respuesta anterior)
def bifurcacion_phi_T(params_dict, y0_pro, y0_res, tau, t_final=500, num_pasos=50):
    phi_T_range = np.linspace(0.02, 0.15, num_pasos)
    T_pro, T_res = [], []
    for phi_T in phi_T_range:
        params = (params_dict['gamma_T'], params_dict['gamma_6'], params_dict['gamma_I'],
                  params_dict['alpha_T'], params_dict['alpha_6'],
                  params_dict['delta_T'], params_dict['delta_6'], params_dict['delta_I'],
                  phi_T, params_dict['phi_6'], params_dict['K'], params_dict['n'])
        sol_pro = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                            (0, t_final), y0_pro, method='RK45', t_eval=[t_final])
        sol_res = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                            (0, t_final), y0_res, method='RK45', t_eval=[t_final])
        T_pro.append(sol_pro.y[0, -1])
        T_res.append(sol_res.y[0, -1])
    plt.figure(figsize=(8,5))
    plt.plot(phi_T_range, T_pro, 'b.-', label='Rama proinflamatoria')
    plt.plot(phi_T_range, T_res, 'g.-', label='Rama resolutiva')
    plt.axvline(x=0.08, color='r', linestyle='--', label='Punto de bifurcación (aprox)')
    plt.xlabel('φ_T (supresión TNF-α por IL-10) [1/(nM·h)]')
    plt.ylabel('TNF-α estacionario (nM)')
    plt.title('Diagrama de bifurcación variando φ_T')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def diagrama_dispersion(params_dict, y0_res, tau, phi_T_range=(0.08, 0.15), num=30):
    phi_vals = np.linspace(phi_T_range[0], phi_T_range[1], num)
    T_vals = []
    for phi_T in phi_vals:
        params = (params_dict['gamma_T'], params_dict['gamma_6'], params_dict['gamma_I'],
                  params_dict['alpha_T'], params_dict['alpha_6'],
                  params_dict['delta_T'], params_dict['delta_6'], params_dict['delta_I'],
                  phi_T, params_dict['phi_6'], params_dict['K'], params_dict['n'])
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                        (0, 500), y0_res, method='RK45', t_eval=[500])
        T_vals.append(sol.y[0, -1])
    def lineal(x, a, b): return a*x + b
    popt, _ = curve_fit(lineal, phi_vals, T_vals)
    a, b = popt
    T_pred = lineal(phi_vals, a, b)
    ss_res = np.sum((T_vals - T_pred)**2)
    ss_tot = np.sum((T_vals - np.mean(T_vals))**2)
    r2 = 1 - ss_res/ss_tot
    plt.figure(figsize=(6,5))
    plt.scatter(phi_vals, T_vals, color='green', label='Datos')
    plt.plot(phi_vals, T_pred, 'k--', label=f'Ajuste lineal (R² = {r2:.3f})')
    plt.xlabel('φ_T')
    plt.ylabel('TNF-α estacionario (nM)')
    plt.title('Rama resolutiva: correlación φ_T vs TNF-α')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    print(f"Ecuación: TNF-α = {a:.4f} * φ_T + {b:.4f}")
    print(f"R² = {r2:.4f}")

def analisis_sensibilidad(params_dict, y0, tau, t_final=72, N=100):
    problema = {
        'num_vars': 5,
        'names': ['gamma_I', 'phi_T', 'phi_6', 'alpha_T', 'alpha_6'],
        'bounds': [[0.2, 1.5], [0.02, 0.15], [0.01, 0.1], [0.05, 0.3], [0.05, 0.25]]
    }
    param_values = saltelli.sample(problema, N, calc_second_order=False)
    print(f"Evaluando {len(param_values)} simulaciones...")
    Y = np.zeros(len(param_values))
    for i, vals in enumerate(param_values):
        gamma_I, phi_T, phi_6, alpha_T, alpha_6 = vals
        params = (params_dict['gamma_T'], params_dict['gamma_6'], gamma_I,
                  alpha_T, alpha_6,
                  params_dict['delta_T'], params_dict['delta_6'], params_dict['delta_I'],
                  phi_T, phi_6, params_dict['K'], params_dict['n'])
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, tau),
                        (0, t_final), y0, method='RK45', t_eval=[t_final])
        Y[i] = sol.y[0, -1]
    Si = sobol.analyze(problema, Y, calc_second_order=False, print_to_console=False)
    df_sens = pd.DataFrame({
        'Parámetro': problema['names'],
        'S1 (efecto principal)': Si['S1'],
        'ST (efecto total)': Si['ST'],
        'ST_conf': Si['ST_conf']
    })
    print("\nÍndices de Sobol (efecto sobre TNF-α final):")
    print(df_sens.to_string(index=False))
    plt.figure(figsize=(8,4))
    x = np.arange(len(problema['names']))
    plt.bar(x - 0.2, Si['S1'], 0.4, label='S1 (principal)', alpha=0.7)
    plt.bar(x + 0.2, Si['ST'], 0.4, label='ST (total)', alpha=0.7)
    plt.xticks(x, problema['names'])
    plt.ylabel('Índice de Sobol')
    plt.title('Análisis de sensibilidad global')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    return Si

# Crear las pestañas no interactivas
pestanya_bifurcacion = crear_pestanya_no_interactiva('Bifurcación', bifurcacion_phi_T)
pestanya_dispersion = crear_pestanya_no_interactiva('Dispersión', diagrama_dispersion)
pestanya_sensibilidad = crear_pestanya_no_interactiva('Sensibilidad', analisis_sensibilidad)

# Construir el Tab
tab = widgets.Tab()
tab.children = [
    out_basal,
    widgets.VBox([dosis_widget, tiempo_dosis_widget, out_bolus]),
    widgets.VBox([factor_widget, out_inhibicion]),
    pestanya_bifurcacion,
    pestanya_dispersion,
    pestanya_sensibilidad
]
tab.set_title(0, 'Basal')
tab.set_title(1, 'IL-10 bolus')
tab.set_title(2, 'Inhibición TNF')
tab.set_title(3, 'Bifurcación')
tab.set_title(4, 'Dispersión')
tab.set_title(5, 'Sensibilidad')

# Mostrar todo
display(widgets.VBox([slider_panel, tab]))

In [ ]:
# -*- coding: utf-8 -*-
"""
Modelo corregido con nomenclatura del artículo Arishi et al. (2026) PLOS ONE.
Todos los parámetros siguen la Tabla 1.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
from SALib.sample import saltelli
from SALib.analyze import sobol
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------
# MODELO CORREGIDO (CON NOMENCLATURA DEL ARTÍCULO)
# ----------------------------------------------------------------------
def model_corregido(t, y, params, tau):
    T, I6, I = y  # TNF-α, IL-6, IL-10
    (α_T, α_6, α_10,                # producciones basales
     k_T6, k_6T,                      # activación cruzada
     β_T, β_6, β_10,                  # degradaciones
     γ_T10, γ_610,                     # supresión por IL-10
     k_10T, k_106,                      # inducción de IL-10 por TNF e IL-6
     K, n) = params                     # Hill: constante y cooperatividad

    # TNF-α
    dT_dt = α_T + k_T6 * I6 - β_T * T - γ_T10 * I * T
    # IL-6
    dI6_dt = α_6 + k_6T * T - β_6 * I6 - γ_610 * I * I6
    # IL-10: producción basal + inducción con retraso y Hill (usando SUMA)
    if t >= tau:
        S = T + I6
        Hill = (S**n) / (K**n + S**n)
        induccion = (k_10T * T + k_106 * I6) * Hill
    else:
        induccion = 0.0
    dI_dt = α_10 + induccion - β_10 * I

    return [dT_dt, dI6_dt, dI_dt]

# ----------------------------------------------------------------------
# PARÁMETROS BASE TOMADOS DE LA TABLA 1 (valores por defecto)
# ----------------------------------------------------------------------
# Producciones basales (nM·hr⁻¹)
α_T_base   = 0.02
α_6_base   = 0.01
α_10_base  = 0.005

# Activación cruzada (nM⁻¹·hr⁻¹)
k_T6_base  = 0.45   # IL-6 → TNF-α
k_6T_base  = 0.40   # TNF-α → IL-6

# Degradaciones (hr⁻¹)
β_T_base   = 1.8
β_6_base   = 1.2
β_10_base  = 0.35

# Supresión por IL-10 (nM⁻¹·hr⁻¹)
γ_T10_base = 0.10   # sobre TNF-α
γ_610_base = 0.06   # sobre IL-6

# Inducción de IL-10 (nM⁻¹·hr⁻¹)
k_10T_base = 0.15   # vía TNF-α
k_106_base = 0.12   # vía IL-6

# Hill
K_base     = 0.3    # nM
n_base     = 5.0

# Retraso (hr)
τ_base     = 18.0

# Condiciones iniciales (nM)
T0_base    = 0.1
I60_base   = 0.05
I0_base    = 0.01

# ----------------------------------------------------------------------
# FUNCIÓN PRINCIPAL DE SIMULACIÓN
# ----------------------------------------------------------------------
def ejecutar_simulacion(
    α_T, α_6, α_10,
    k_T6, k_6T,
    β_T, β_6, β_10,
    γ_T10, γ_610,
    k_10T, k_106,
    K, n,
    τ, T0, I60, I0,
    tipo='basal',
    dosis=0.05, tiempo_dosis=24, factor_inhibicion=0.5
):
    params = [α_T, α_6, α_10,
              k_T6, k_6T,
              β_T, β_6, β_10,
              γ_T10, γ_610,
              k_10T, k_106,
              K, n]
    y0 = [T0, I60, I0]
    t_span = (0, 72)
    t_eval = np.linspace(0, 72, 500)

    fig, ax = plt.subplots(figsize=(10, 5))

    if tipo == 'basal':
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, τ),
                        t_span, y0, t_eval=t_eval, method='RK45')
        T, I6, I = sol.y
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.axvspan(6, 24, alpha=0.1, color='gray', label='Ventana proinflamatoria')
        ax.axvspan(36, 60, alpha=0.1, color='lightgreen', label='Ventana resolución')
        ax.axvline(τ, color='k', ls='--', lw=1, label=f'Retraso τ={τ}h')
        ax.set_title('Simulación basal (parámetros del artículo)')

    elif tipo == 'bolus':
        def model_with_bolus(t, y):
            if abs(t - tiempo_dosis) < 1e-2:
                y = list(y)
                y[2] += dosis
            return model_corregido(t, y, params, τ)
        sol = solve_ivp(lambda t, y: model_with_bolus(t, y),
                        t_span, y0, t_eval=t_eval, method='RK45', max_step=0.1)
        T, I6, I = sol.y
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.axvline(tiempo_dosis, color='orange', ls=':', lw=2, label=f'Dosis IL-10 (t={tiempo_dosis}h)')
        ax.set_title(f'Dosis de IL-10 ({dosis} nM a las {tiempo_dosis} h)')

    elif tipo == 'inhibicion':
        # Inhibición de TNF-α: reducimos su producción basal y activación
        α_T_inh = α_T * factor_inhibicion
        k_T6_inh = k_T6 * factor_inhibicion
        params_inh = [α_T_inh, α_6, α_10,
                      k_T6_inh, k_6T,
                      β_T, β_6, β_10,
                      γ_T10, γ_610,
                      k_10T, k_106,
                      K, n]
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params_inh, τ),
                        t_span, y0, t_eval=t_eval, method='RK45')
        T, I6, I = sol.y
        ax.plot(sol.t, T, 'r-', lw=2, label='TNF-α')
        ax.plot(sol.t, I6, 'b-', lw=2, label='IL-6')
        ax.plot(sol.t, I, 'g-', lw=2, label='IL-10')
        ax.set_title(f'Inhibición TNF-α ({factor_inhibicion*100:.0f}%)')

    ax.set_xlabel('Tiempo (h)')
    ax.set_ylabel('Concentración (nM)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# ----------------------------------------------------------------------
# CONSTRUCCIÓN DE LOS WIDGETS (con nombres y rangos adecuados)
# ----------------------------------------------------------------------
sliders = {
    'α_T':   widgets.FloatSlider(value=α_T_base,   min=0.0,   max=0.1,   step=0.002, description='α_T (basal TNF)'),
    'α_6':   widgets.FloatSlider(value=α_6_base,   min=0.0,   max=0.05,  step=0.001, description='α_6 (basal IL-6)'),
    'α_10':  widgets.FloatSlider(value=α_10_base,  min=0.0,   max=0.02,  step=0.001, description='α_10 (basal IL-10)'),
    'k_T6':  widgets.FloatSlider(value=k_T6_base,  min=0.0,   max=0.8,   step=0.01,  description='k_T6 (IL-6→TNF)'),
    'k_6T':  widgets.FloatSlider(value=k_6T_base,  min=0.0,   max=0.8,   step=0.01,  description='k_6T (TNF→IL-6)'),
    'β_T':   widgets.FloatSlider(value=β_T_base,   min=0.5,   max=2.5,   step=0.05,  description='β_T (degr TNF)'),
    'β_6':   widgets.FloatSlider(value=β_6_base,   min=0.5,   max=2.0,   step=0.05,  description='β_6 (degr IL-6)'),
    'β_10':  widgets.FloatSlider(value=β_10_base,  min=0.1,   max=0.8,   step=0.02,  description='β_10 (degr IL-10)'),
    'γ_T10': widgets.FloatSlider(value=γ_T10_base, min=0.02,  max=0.2,   step=0.005, description='γ_T10 (sup TNF)'),
    'γ_610': widgets.FloatSlider(value=γ_610_base, min=0.01,  max=0.15,  step=0.005, description='γ_610 (sup IL-6)'),
    'k_10T': widgets.FloatSlider(value=k_10T_base, min=0.0,   max=0.4,   step=0.01,  description='k_10T (TNF→IL-10)'),
    'k_106': widgets.FloatSlider(value=k_106_base, min=0.0,   max=0.3,   step=0.01,  description='k_106 (IL-6→IL-10)'),
    'K':     widgets.FloatSlider(value=K_base,     min=0.1,   max=1.0,   step=0.05,  description='K (Hill)'),
    'n':     widgets.FloatSlider(value=n_base,     min=1.0,   max=8.0,   step=0.2,   description='n (coop)'),
    'τ':     widgets.FloatSlider(value=τ_base,     min=0.0,   max=48.0,  step=1.0,   description='τ (retraso)'),
    'T0':    widgets.FloatSlider(value=T0_base,    min=0.0,   max=0.5,   step=0.01,  description='TNF₀'),
    'I60':   widgets.FloatSlider(value=I60_base,   min=0.0,   max=0.5,   step=0.01,  description='IL-6₀'),
    'I0':    widgets.FloatSlider(value=I0_base,    min=0.0,   max=0.1,   step=0.001, description='IL-10₀')
}

# Organizar sliders en filas de 5
slider_items = list(sliders.items())
rows = []
for i in range(0, len(slider_items), 5):
    row = widgets.HBox([slider_items[j][1] for j in range(i, min(i+5, len(slider_items)))])
    rows.append(row)
slider_panel = widgets.VBox(rows)

# ----------------------------------------------------------------------
# PESTAÑAS INTERACTIVAS
# ----------------------------------------------------------------------
args_comunes = {name: slider for name, slider in sliders.items()}

# Pestaña basal
out_basal = widgets.interactive_output(
    ejecutar_simulacion,
    {**args_comunes, 'tipo': widgets.fixed('basal')}
)

# Pestaña bolus
dosis_widget = widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.01, description='Dosis (nM)')
tiempo_dosis_widget = widgets.FloatSlider(value=24, min=0, max=72, step=1, description='Tiempo dosis (h)')
out_bolus = widgets.interactive_output(
    ejecutar_simulacion,
    {**args_comunes,
     'tipo': widgets.fixed('bolus'),
     'dosis': dosis_widget,
     'tiempo_dosis': tiempo_dosis_widget}
)

# Pestaña inhibición
factor_widget = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Factor inhibición')
out_inhibicion = widgets.interactive_output(
    ejecutar_simulacion,
    {**args_comunes,
     'tipo': widgets.fixed('inhibicion'),
     'factor_inhibicion': factor_widget}
)

# ----------------------------------------------------------------------
# FUNCIONES PARA ANÁLISIS AVANZADOS (bifurcación, dispersión, sensibilidad)
# ----------------------------------------------------------------------
def bifurcacion_γ_T10(params_dict, y0_pro, y0_res, τ, t_final=500, num=50):
    γ_range = np.linspace(0.02, 0.15, num)
    T_pro, T_res = [], []
    for γ in γ_range:
        params = [params_dict['α_T'], params_dict['α_6'], params_dict['α_10'],
                  params_dict['k_T6'], params_dict['k_6T'],
                  params_dict['β_T'], params_dict['β_6'], params_dict['β_10'],
                  γ, params_dict['γ_610'],
                  params_dict['k_10T'], params_dict['k_106'],
                  params_dict['K'], params_dict['n']]
        sol_pro = solve_ivp(lambda t, y: model_corregido(t, y, params, τ),
                            (0, t_final), y0_pro, method='RK45', t_eval=[t_final])
        sol_res = solve_ivp(lambda t, y: model_corregido(t, y, params, τ),
                            (0, t_final), y0_res, method='RK45', t_eval=[t_final])
        T_pro.append(sol_pro.y[0, -1])
        T_res.append(sol_res.y[0, -1])
    plt.figure(figsize=(8,5))
    plt.plot(γ_range, T_pro, 'b.-', label='Rama proinflamatoria')
    plt.plot(γ_range, T_res, 'g.-', label='Rama resolutiva')
    plt.axvline(x=0.06, color='r', linestyle='--', label='Punto bifurcación ≈0.06')
    plt.xlabel('γ_T10 (supresión TNF-α por IL-10) [nM⁻¹·hr⁻¹]')
    plt.ylabel('TNF-α estacionario (nM)')
    plt.title('Diagrama de bifurcación (variando γ_T10)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def diagrama_dispersion(params_dict, y0_res, τ, γ_range=(0.08, 0.15), num=30):
    γ_vals = np.linspace(γ_range[0], γ_range[1], num)
    T_vals = []
    for γ in γ_vals:
        params = [params_dict['α_T'], params_dict['α_6'], params_dict['α_10'],
                  params_dict['k_T6'], params_dict['k_6T'],
                  params_dict['β_T'], params_dict['β_6'], params_dict['β_10'],
                  γ, params_dict['γ_610'],
                  params_dict['k_10T'], params_dict['k_106'],
                  params_dict['K'], params_dict['n']]
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, τ),
                        (0, 500), y0_res, method='RK45', t_eval=[500])
        T_vals.append(sol.y[0, -1])
    def lineal(x, a, b): return a*x + b
    popt, _ = curve_fit(lineal, γ_vals, T_vals)
    a, b = popt
    T_pred = lineal(γ_vals, a, b)
    ss_res = np.sum((T_vals - T_pred)**2)
    ss_tot = np.sum((T_vals - np.mean(T_vals))**2)
    r2 = 1 - ss_res/ss_tot
    plt.figure(figsize=(6,5))
    plt.scatter(γ_vals, T_vals, color='green', label='Datos')
    plt.plot(γ_vals, T_pred, 'k--', label=f'Ajuste lineal (R² = {r2:.3f})')
    plt.xlabel('γ_T10')
    plt.ylabel('TNF-α estacionario (nM)')
    plt.title('Rama resolutiva: correlación γ_T10 vs TNF-α')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    print(f"Ecuación: TNF-α = {a:.4f} * γ_T10 + {b:.4f}")
    print(f"R² = {r2:.4f}")

def analisis_sensibilidad(params_dict, y0, τ, t_final=72, N=50):
    problema = {
        'num_vars': 6,
        'names': ['γ_T10', 'k_10T', 'k_106', 'n', 'K', 'α_T'],
        'bounds': [[0.02, 0.15], [0.05, 0.25], [0.04, 0.20], [2, 8], [0.1, 0.5], [0.005, 0.05]]
    }
    param_values = saltelli.sample(problema, N, calc_second_order=False)
    print(f"Evaluando {len(param_values)} simulaciones...")
    Y = np.zeros(len(param_values))
    for i, vals in enumerate(param_values):
        γ_T10, k_10T, k_106, n, K, α_T = vals
        params = [α_T, params_dict['α_6'], params_dict['α_10'],
                  params_dict['k_T6'], params_dict['k_6T'],
                  params_dict['β_T'], params_dict['β_6'], params_dict['β_10'],
                  γ_T10, params_dict['γ_610'],
                  k_10T, k_106,
                  K, n]
        sol = solve_ivp(lambda t, y: model_corregido(t, y, params, τ),
                        (0, t_final), y0, method='RK45', t_eval=[t_final])
        Y[i] = sol.y[0, -1]
    Si = sobol.analyze(problema, Y, calc_second_order=False, print_to_console=False)
    df_sens = pd.DataFrame({
        'Parámetro': problema['names'],
        'S1 (efecto principal)': Si['S1'],
        'ST (efecto total)': Si['ST'],
        'ST_conf': Si['ST_conf']
    })
    print("\nÍndices de Sobol (efecto sobre TNF-α final):")
    print(df_sens.to_string(index=False))
    plt.figure(figsize=(8,4))
    x = np.arange(len(problema['names']))
    plt.bar(x - 0.2, Si['S1'], 0.4, label='S1 (principal)', alpha=0.7)
    plt.bar(x + 0.2, Si['ST'], 0.4, label='ST (total)', alpha=0.7)
    plt.xticks(x, problema['names'])
    plt.ylabel('Índice de Sobol')
    plt.title('Análisis de sensibilidad global')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    return Si

# ----------------------------------------------------------------------
# CONSTRUCCIÓN DE PESTAÑAS NO INTERACTIVAS (con botones)
# ----------------------------------------------------------------------
def crear_pestanya_analisis(titulo, funcion, y0_res=None):
    button = widgets.Button(description=f"Ejecutar {titulo}")
    output = widgets.Output()
    def on_click(b):
        with output:
            clear_output(wait=True)
            params_dict = {k: sliders[k].value for k in
                           ['α_T','α_6','α_10','k_T6','k_6T','β_T','β_6','β_10',
                            'γ_T10','γ_610','k_10T','k_106','K','n']}
            τ = sliders['τ'].value
            y0 = [sliders['T0'].value, sliders['I60'].value, sliders['I0'].value]
            if titulo == 'Bifurcación':
                y0_pro = [0.5, 0.3, 0.01]   # condición proinflamatoria
                y0_res = [0.05, 0.03, 0.5]  # condición resolutiva
                funcion(params_dict, y0_pro, y0_res, τ)
            elif titulo == 'Dispersión':
                y0_res = [0.05, 0.03, 0.5] if y0_res is None else y0_res
                funcion(params_dict, y0_res, τ)
            elif titulo == 'Sensibilidad':
                funcion(params_dict, y0, τ, t_final=72, N=50)
    button.on_click(on_click)
    return widgets.VBox([button, output])

pest_bifurc = crear_pestanya_analisis('Bifurcación', bifurcacion_γ_T10)
pest_disp   = crear_pestanya_analisis('Dispersión', diagrama_dispersion)
pest_sens   = crear_pestanya_analisis('Sensibilidad', analisis_sensibilidad)

# ----------------------------------------------------------------------
# CONFIGURACIÓN DE LAS PESTAÑAS PRINCIPALES
# ----------------------------------------------------------------------
tab = widgets.Tab()
tab.children = [
    out_basal,
    widgets.VBox([dosis_widget, tiempo_dosis_widget, out_bolus]),
    widgets.VBox([factor_widget, out_inhibicion]),
    pest_bifurc,
    pest_disp,
    pest_sens
]
tab.set_title(0, 'Basal')
tab.set_title(1, 'IL-10 bolus')
tab.set_title(2, 'Inhibición TNF')
tab.set_title(3, 'Bifurcación')
tab.set_title(4, 'Dispersión')
tab.set_title(5, 'Sensibilidad')

# Mostrar todo
display(widgets.VBox([slider_panel, tab]))